# ML replacement for the fuzzy-AHP industry ranking

Paper: Srinivas & Singh (2018), *Impact assessment of industrial wastewater discharge in a river basin using interval-valued fuzzy group decision-making and spatial approach*, Environ Dev Sustain 20:2373–2397.

| Paper (fuzzy) | This notebook (ML) |
|---|---|
| 3 experts rate 33 sources on C1–C10 (1/3/5/7/9) | Measured source conditions (flow, inlet/outlet, treatment, rain, river flow) |
| Pairwise AHP matrices + consistency check | Training rows: source × sampling event × pollutant |
| Interval-valued fuzzy number `[(p',p); q; (r,r')]` | Quantile prediction interval |
| Defuzzified local weight per criterion | Predicted river change, groundwater change, load, violation probability |
| Weighted sum (Eq. 10) with expert criterion weights | Explicit policy rule (`wqml.POLICY`, `TOXICITY`) — set by experts, not learned |

Criteria mapping: C1 → `outlet_load_kg_day`, C2 → treatment efficiency (inlet vs outlet features), C3/C4 → `violation`, `exceedance_ratio` by season, C6/C7 → `river_change` by season, C8/C9 → `gw_change` by season, C10 → `TOXICITY` weights (policy).

**Until `data/measurements.csv` exists this runs on SYNTHETIC data. Numbers below mean nothing then — it only proves the pipeline works.**

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import wqml

DATA = Path("data/measurements.csv")   # model rows (data/template.csv format), or
SAMPLES = Path("data/samples.csv")     # raw lab sheet (data/samples_template.csv format) -> reshaped
STATIONS = Path("data/stations.csv")   # station register; falls back to the paper's Table 1 register
ECHO_DATA = wqml.ECHO / "echo_dmr.csv"  # stand-in: real US EPA discharge reports (wqml.load_echo)
SYNTHETIC = STAND_IN = False
if DATA.exists():
    raw = pd.read_csv(DATA)
elif SAMPLES.exists():
    reg = pd.read_csv(STATIONS if STATIONS.exists() else wqml.PAPER / "stations.csv")
    raw = wqml.reshape(pd.read_csv(SAMPLES), reg)
elif ECHO_DATA.exists():
    raw = pd.read_csv(ECHO_DATA)
    STAND_IN = True
    print("NOTE: US EPA ECHO stand-in data (US tanneries/textile/paper/meat plants), not Unnao.")
else:
    raw = wqml.synthetic()
    SYNTHETIC = True
    print("WARNING: synthetic data. Results are placeholders.")
raw.shape

## 0. What the paper actually published (`data/paper/`)
39 measurement rows from 8 sources (Table 2 + numbers in the text), no dates, no upstream/downstream or well pairs → **not trainable**. Useful for units, magnitudes and limit checks. Audit duplicates are expected (pre/post-monsoon rows share a blank date).

In [ ]:
paper = wqml.prepare(pd.read_csv(wqml.PAPER / "measurements.csv"))
display(wqml.audit(paper).T)
paper.dropna(subset=["limit"]).pivot_table(index=["source_id", "season"], columns="pollutant",
                                           values="exceedance_ratio", dropna=False).round(2)

## 1. Audit
Schema errors raise. Quality problems are counted — fix them at the source, not in the model.

In [ ]:
wqml.audit(raw)

## 2. Targets
Physical outcomes (index.html *targets the model must learn*). `downstream` and `near_well` are labels, never features.

In [ ]:
df = wqml.prepare(raw)
# Train only targets with labels from >= 5 sources (grouped CV needs whole sources to hold out).
n_src = {t: df.dropna(subset=[t])["source_id"].nunique() for t in wqml.TARGETS}
targets = [t for t, n in n_src.items() if n >= 5]
print("sources per target:", n_src, "-> training", targets)
df[targets + ["exceedance_ratio"]].describe().T

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
for a, t in zip(ax, [t for t in targets if wqml.TARGETS[t][0] == "reg"]):
    df[t].clip(df[t].quantile(.01), df[t].quantile(.99)).hist(bins=60, ax=a)
    a.set_title(t)
plt.tight_layout()
df.groupby("pollutant")["violation"].mean().sort_values().plot.barh(title="violation rate by pollutant", figsize=(5, 4));

In [ ]:
# Missingness per column: large gaps here decide which targets are trainable at all.
df[wqml.COLUMNS].isna().mean().sort_values(ascending=False).head(10)

## 3. Compare models (source-grouped CV)
Whole sources held out per fold → score = performance on an **unseen industry**. Elastic Net / logistic is the baseline every other model must beat.

In [ ]:
results = {t: wqml.compare(df, t) for t in targets}
for t, r in results.items():
    print(t); display(r.round(3))

## 4. Pick the simplest model that holds up
Pick per target: lowest MAE (regression) or Brier (classification); prefer the baseline when the gap is small (< 5%).

In [ ]:
def pick(r, task):
    col = "MAE" if task == "reg" else "Brier"
    base = "elastic_net" if task == "reg" else "logistic"
    best = r[col].idxmin()
    return base if r.loc[base, col] <= r.loc[best, col] * 1.05 else best

chosen = {t: pick(results[t], wqml.TARGETS[t][0]) for t in targets}
chosen

## 5. Strict check: future dates **and** unseen sources

In [ ]:
pd.DataFrame({t: wqml.future_holdout(df, t, m) for t, m in chosen.items()}).T

## 6. Uncertainty
Replaces the fuzzy interval. Out-of-fold 80% quantile interval; coverage should be ≈ 0.80. Far off → intervals are not trustworthy.

In [ ]:
names = {"river_change": "river", "gw_change": "groundwater", "violation": "violation"}
oof = {names[t]: wqml.oof_predictions(df, t, chosen[t]) for t in names if t in chosen}
{k: round(wqml.coverage(v), 3) for k, v in oof.items() if "hi" in v}

## 7. Ranking (policy layer)
Replaces paper Eq. 10. `POLICY` and `TOXICITY` in `wqml.py` are **placeholders** — must be agreed with domain/regulatory experts. Higher risk = higher priority; ranked within each industrial area like Table 10.

In [ ]:
ranking = wqml.rank_sources(df, oof)
ranking

## 8. Sanity check against the paper
Paper: low fuzzy weight = more polluting, so expect **negative** Spearman ρ between ML risk and fuzzy weight. Only meaningful with real data whose `source_id` follows `<area>-S<n>` from Table 10. Disagreement is not automatically an ML error — experts had 2012 snapshots, the model has repeated measurements.

In [ ]:
cmp = ranking.join(wqml.paper_ranking(), how="inner")
if SYNTHETIC or STAND_IN or cmp.empty:
    print("no real Unnao sources in this data -> nothing to compare")
else:
    display(cmp.groupby("industrial_area").apply(lambda g: g["risk"].corr(g["fuzzy_weight"], method="spearman")).rename("spearman_rho"))

## Before trusting any output
- [ ] Real data in `data/measurements.csv`, repeated monthly across seasons (≥ 1 year), stable `source_id`s
- [ ] Audit table all zeros or explained
- [ ] Baseline beaten on source-grouped CV *and* future holdout
- [ ] Interval coverage ≈ nominal
- [ ] `POLICY` / `TOXICITY` signed off by domain experts
- [ ] Hyperparameter tuning with grouped nested CV (currently fixed)